# E1 — Test-time topology swap diagnostic
## Does a trained graph ranker actually use the topology supplied at inference?

This notebook implements the first additional diagnostic proposed during the pre-JMS review.

### Question

The primary experiment showed that models trained with the recorded topology and with semantic--topology permutations both outperform a separately trained no-edge GNN. That result establishes that allowing cross-node information is useful, but it does not directly test whether a model trained on the recorded topology relies on the particular held-out topology presented at inference.

E1 changes **only the held-out product graph at test time** while keeping the trained model fixed.

### Forward swap: model trained on recorded topology

For every LOPO fold and neural-training seed, one `RealGNN` is trained exactly as in the canonical experiment and then evaluated on the same held-out battery under:

1. the recorded physical graph (`real -> real`, matched);
2. each of the five deterministic semantic--topology permutations (`real -> perm1...perm5`);
3. an empty graph (`real -> no-edge`).

The five permuted evaluations are averaged within each fold/seed before product-level inference.

### Reverse swap: models trained on permuted topology

For every LOPO fold, seed, and permutation ID, the corresponding `ShuffledGNN` is trained exactly as in the canonical experiment and evaluated on:

1. its matched permutation (`perm_r -> perm_r`);
2. the recorded physical graph (`perm_r -> real`).

This reverse direction helps distinguish sensitivity to physical semantic alignment from generic train--test topology mismatch.

### Important implementation note

The original canonical notebook did **not** persist trained checkpoints. Therefore, this notebook reproduces the original RealGNN and five permutation-trained models using the same folds, seeds, architecture, hyperparameters, negative-sampling procedure, and topology-permutation seeds. Immediately after each model is trained, the topology is changed **only for inference**. Checkpoints are saved here so the experiment is resumable.

This is a new robustness diagnostic, not a preregistered experiment. Its contrasts and aggregation rules are fixed in the notebook before E1 results are inspected.

### Primary E1 Hits@1 contrasts

At the held-out-product level (`n=10`):

- **A. Real-trained: real test − five-permutation test mean.**
- **B. Real-trained: real test − no-edge test.**
- **C. Permutation-trained: matched-permutation test − real test.**

The held-out battery remains the independent unit. Neural seeds and permutation realizations are averaged within product and are not treated as additional independent observations. Exact paired sign-flip tests govern inference; percentile-bootstrap confidence intervals describe effect magnitude. Holm adjustment is applied across the three E1 Hits@1 contrasts. Hits@3 and MRR are supporting endpoints.

A positive A indicates that a model trained on recorded graphs benefits from correct semantic--topological alignment on an unseen product. A positive B indicates reliance on graph-branch cross-node information at inference. Contrast C diagnoses sensitivity of permutation-trained models to replacing their learned topology convention with the recorded topology; it should not by itself be interpreted as evidence for or against physical correctness.


In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRAIN_FITS = 10 * len(SEEDS) * (1 + len(SHUFFLE_IDS))

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned E1 training fits:", N_TRAIN_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product, progress
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr, progress):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
            progress,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    current, positive, negative, progress = [], [], [], []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)
        progress.append(
            t / max(1, n_ops - 1)
        )

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
        np.asarray(progress, dtype=np.float32),
    )


def build_gnn(vocab):
    model = GNNOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
    )
    return model.to(DEVICE)


def train_one_gnn_condition(
    train_batteries,
    test_battery,
    seed,
    graph_mode,
    shuffle_id=0,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode=graph_mode,
            shuffle_id=shuffle_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_gnn(vocab)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    # Using the same training seed across paired graph conditions
    # keeps initialization, training order, and negative sampling paired.
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg, prog = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            prog_t = torch.as_tensor(
                prog,
                dtype=torch.float32,
                device=DEVICE,
            ).unsqueeze(-1)

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
                prog_t,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )

            neg_z = z[neg_t.reshape(-1)]

            prog_rep = (
                prog_t[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, 1)
            )

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
                prog_rep,
            ).view(len(cur), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model, packs[test_battery], vocab

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            progress = torch.full(
                (len(candidate_indices), 1),
                t / max(1, n_ops - 1),
                dtype=torch.float32,
                device=DEVICE,
            )

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
                progress,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Exact-capacity validity helper

def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

## 13. E1 helpers and resumable checkpointing

The canonical training function above is reused without modification. The helper below saves one checkpoint for each `(held-out battery, training seed, training topology)` combination. This avoids repeating expensive training if Colab disconnects.

Every topology swap is performed after training by rebuilding only the **held-out** battery pack with a different `graph_mode`; model parameters are not updated during swapped evaluation.


In [ ]:
# 13. E1 output paths, checkpoint helpers, and exact-test utilities

import itertools
import shutil

OUT = Path("/content/battery_test_time_topology_swap_lopo")
CKPT_DIR = OUT / "checkpoints"
OUT.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

PARTIAL = OUT / "test_time_topology_swap_partial.csv"

def checkpoint_path(test_battery, seed, train_graph_mode, train_shuffle_id):
    return CKPT_DIR / (
        f"battery_{int(test_battery):02d}"
        f"__seed_{int(seed)}"
        f"__train_{train_graph_mode}"
        f"__shuffle_{int(train_shuffle_id)}.pt"
    )

def train_or_load_gnn(
    train_batteries,
    test_battery,
    seed,
    train_graph_mode,
    train_shuffle_id=0,
):
    # Reproduce one canonical training condition, or load its E1 checkpoint.
    vocab = build_fold_vocab(train_batteries)
    model = build_gnn(vocab)
    ckpt = checkpoint_path(
        test_battery,
        seed,
        train_graph_mode,
        train_shuffle_id,
    )

    if ckpt.exists():
        payload = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(payload["model_state_dict"])
        model.to(DEVICE)
        return model, vocab, True

    model, _, returned_vocab = train_one_gnn_condition(
        train_batteries=train_batteries,
        test_battery=test_battery,
        seed=seed,
        graph_mode=train_graph_mode,
        shuffle_id=train_shuffle_id,
    )

    assert returned_vocab == vocab

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "test_battery": int(test_battery),
            "seed": int(seed),
            "train_graph_mode": str(train_graph_mode),
            "train_shuffle_id": int(train_shuffle_id),
            "epochs": int(EPOCHS),
            "learning_rate": float(LR),
            "weight_decay": float(WEIGHT_DECAY),
        },
        ckpt,
    )
    return model, vocab, False


def exact_two_sided_sign_flip(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    assert n > 0

    observed = abs(values.mean())
    extreme = 0
    total = 2 ** n

    for signs in itertools.product((-1.0, 1.0), repeat=n):
        stat = abs(np.mean(values * np.asarray(signs)))
        if stat >= observed - 1e-15:
            extreme += 1

    return extreme / total


def bootstrap_mean_ci(values, n_boot=20000, seed=20261220):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    draws = rng.choice(
        values,
        size=(n_boot, len(values)),
        replace=True,
    ).mean(axis=1)
    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    m = len(pvalues)
    order = np.argsort(pvalues)
    adjusted_sorted = np.empty(m, dtype=float)
    running = 0.0

    for rank, idx in enumerate(order):
        candidate = (m - rank) * pvalues[idx]
        running = max(running, candidate)
        adjusted_sorted[rank] = min(1.0, running)

    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]
    return out


print("E1 output directory:", OUT)
print("E1 checkpoints:", CKPT_DIR)


## 14. Sanity checks

This cell verifies that the five held-out permutation packs are distinct and that changing the test topology does not alter operation-level semantic attributes. It does not train a model.


In [ ]:
# 14. Topology-swap sanity checks on one fold

TEST_BATTERY = sorted(chains)[0]
train_batteries = [b for b in sorted(chains) if b != TEST_BATTERY]
vocab = build_fold_vocab(train_batteries)

real_pack = build_battery_pack(
    TEST_BATTERY, vocab, graph_mode="real", shuffle_id=0
)
noedge_pack = build_battery_pack(
    TEST_BATTERY, vocab, graph_mode="noedge", shuffle_id=0
)
perm_packs = [
    build_battery_pack(
        TEST_BATTERY, vocab, graph_mode="shuffle", shuffle_id=sid
    )
    for sid in SHUFFLE_IDS
]

for p in [noedge_pack] + perm_packs:
    assert np.array_equal(real_pack["op_categorical"], p["op_categorical"])
    assert np.array_equal(real_pack["op_graph_nodes"], p["op_graph_nodes"])
    assert np.allclose(real_pack["op_numeric"], p["op_numeric"])

fingerprints = [p["edge_index"].tobytes() for p in perm_packs]
assert len(set(fingerprints)) == len(SHUFFLE_IDS)

assert noedge_pack["edge_index"].shape[1] == 0
assert np.allclose(noedge_pack["node_degree"], 0.0)

print("Sanity checks passed for:", battery_names[TEST_BATTERY])
print("Real edges:", real_pack["edge_index"].shape[1])
print("No-edge edges:", noedge_pack["edge_index"].shape[1])
print("Distinct permutation packs:", len(set(fingerprints)))


## 15. Full E1 experiment

Training conditions:

- `RealGNN`: 10 folds × 3 neural seeds = 30 fits.
- `ShuffledGNN`: 10 folds × 3 neural seeds × 5 fixed permutation IDs = 150 fits.

Total: **180 training fits**.

A real-trained model is evaluated on seven held-out graph variants; a permutation-trained model is evaluated on its matched permutation and on the real held-out graph.

The experiment is resumable at both the checkpoint and CSV levels.


In [ ]:
# 15. Run the full test-time topology-swap experiment

if PARTIAL.exists():
    swap_results = pd.read_csv(PARTIAL)
    print("Resuming from", len(swap_results), "completed evaluation rows.")
else:
    swap_results = pd.DataFrame()

battery_ids = sorted(chains)
start_time = time.time()

train_conditions = [
    ("RealGNN", "real", 0),
] + [
    ("ShuffledGNN", "shuffle", sid)
    for sid in SHUFFLE_IDS
]

def eval_already_done(
    df,
    test_battery,
    seed,
    train_graph_mode,
    train_shuffle_id,
    test_graph_mode,
    test_shuffle_id,
):
    if len(df) == 0:
        return False
    mask = (
        (df["battery_id"] == test_battery)
        & (df["seed"] == seed)
        & (df["train_graph_mode"] == train_graph_mode)
        & (df["train_shuffle_id"] == train_shuffle_id)
        & (df["test_graph_mode"] == test_graph_mode)
        & (df["test_shuffle_id"] == test_shuffle_id)
    )
    return bool(mask.any())


for fold_idx, test_battery in enumerate(battery_ids, start=1):
    train_batteries = [b for b in battery_ids if b != test_battery]

    print(
        f"\nFold {fold_idx}/10 — held out: "
        f"{battery_names[test_battery]}"
    )

    for model_label, train_mode, train_sid in train_conditions:
        for seed in SEEDS:

            if train_mode == "real":
                eval_specs = [
                    ("real", 0, "matched_real"),
                    *[
                        ("shuffle", sid, f"swap_to_perm_{sid}")
                        for sid in SHUFFLE_IDS
                    ],
                    ("noedge", 0, "swap_to_noedge"),
                ]
            else:
                eval_specs = [
                    ("shuffle", train_sid, "matched_permutation"),
                    ("real", 0, "swap_to_real"),
                ]

            all_done = all(
                eval_already_done(
                    swap_results,
                    test_battery,
                    seed,
                    train_mode,
                    train_sid,
                    test_mode,
                    test_sid,
                )
                for test_mode, test_sid, _ in eval_specs
            )
            if all_done:
                print(
                    f"  skip completed {model_label} "
                    f"train_shuffle={train_sid} seed={seed}"
                )
                continue

            model, vocab, loaded = train_or_load_gnn(
                train_batteries=train_batteries,
                test_battery=test_battery,
                seed=seed,
                train_graph_mode=train_mode,
                train_shuffle_id=train_sid,
            )

            source = "checkpoint" if loaded else "trained"
            print(
                f"  {source}: {model_label} "
                f"train_shuffle={train_sid} seed={seed}"
            )

            for test_mode, test_sid, eval_label in eval_specs:
                if eval_already_done(
                    swap_results,
                    test_battery,
                    seed,
                    train_mode,
                    train_sid,
                    test_mode,
                    test_sid,
                ):
                    continue

                test_pack = build_battery_pack(
                    test_battery,
                    vocab,
                    graph_mode=test_mode,
                    shuffle_id=test_sid,
                )

                metrics, _ = evaluate_neural_model(
                    model,
                    test_pack,
                    train_batteries,
                )

                row = {
                    "battery_id": int(test_battery),
                    "battery": battery_names[test_battery],
                    "model": model_label,
                    "seed": int(seed),
                    "train_graph_mode": train_mode,
                    "train_shuffle_id": int(train_sid),
                    "test_graph_mode": test_mode,
                    "test_shuffle_id": int(test_sid),
                    "evaluation": eval_label,
                    "hits1": metrics["hits1"],
                    "hits3": metrics["hits3"],
                    "mrr": metrics["mrr"],
                    "seen_signature_rate": metrics["seen_signature_rate"],
                    "parameter_count": count_trainable_parameters(model),
                }

                swap_results = pd.concat(
                    [swap_results, pd.DataFrame([row])],
                    ignore_index=True,
                )
                swap_results.to_csv(PARTIAL, index=False)

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

elapsed = time.time() - start_time

expected_eval_rows = (
    10 * len(SEEDS) * 7
    + 10 * len(SEEDS) * len(SHUFFLE_IDS) * 2
)

print(
    f"\nCompleted/resumed to {len(swap_results)} evaluation rows "
    f"(expected {expected_eval_rows}) in {elapsed/60:.1f} minutes."
)


## 16. Build paired seed-level and product-level summaries

The five test-time permutations are first averaged within each real-trained `(product, seed)` pair. For the reverse direction, the five permutation-trained models are averaged within each `(product, seed)` after computing their matched-permutation and real-test scores.

Only after these within-product averages are formed are the three neural seeds averaged. Statistical inference therefore remains at `n=10` held-out products.


In [ ]:
# 16. Paired E1 summaries

expected_eval_rows = (
    10 * len(SEEDS) * 7
    + 10 * len(SEEDS) * len(SHUFFLE_IDS) * 2
)
assert len(swap_results) == expected_eval_rows, (
    f"E1 incomplete: {len(swap_results)} rows present; "
    f"{expected_eval_rows} expected. Rerun Cell 15."
)

rt = swap_results[
    swap_results["train_graph_mode"] == "real"
].copy()

rt_real = (
    rt[rt["evaluation"] == "matched_real"]
    [["battery_id", "battery", "seed", "hits1", "hits3", "mrr"]]
    .rename(columns={
        "hits1": "rt_real_hits1",
        "hits3": "rt_real_hits3",
        "mrr": "rt_real_mrr",
    })
)

rt_perm = (
    rt[rt["test_graph_mode"] == "shuffle"]
    .groupby(["battery_id", "battery", "seed"], as_index=False)
    .agg(
        rt_perm_hits1=("hits1", "mean"),
        rt_perm_hits3=("hits3", "mean"),
        rt_perm_mrr=("mrr", "mean"),
        rt_perm_hits1_sd=("hits1", "std"),
    )
)

rt_noedge = (
    rt[rt["evaluation"] == "swap_to_noedge"]
    [["battery_id", "battery", "seed", "hits1", "hits3", "mrr"]]
    .rename(columns={
        "hits1": "rt_noedge_hits1",
        "hits3": "rt_noedge_hits3",
        "mrr": "rt_noedge_mrr",
    })
)

realtrain_seed = (
    rt_real
    .merge(rt_perm, on=["battery_id", "battery", "seed"])
    .merge(rt_noedge, on=["battery_id", "battery", "seed"])
)

st = swap_results[
    swap_results["train_graph_mode"] == "shuffle"
].copy()

st_matched = (
    st[st["evaluation"] == "matched_permutation"]
    .groupby(["battery_id", "battery", "seed"], as_index=False)
    .agg(
        st_matched_hits1=("hits1", "mean"),
        st_matched_hits3=("hits3", "mean"),
        st_matched_mrr=("mrr", "mean"),
        st_matched_hits1_sd=("hits1", "std"),
    )
)

st_real = (
    st[st["evaluation"] == "swap_to_real"]
    .groupby(["battery_id", "battery", "seed"], as_index=False)
    .agg(
        st_real_hits1=("hits1", "mean"),
        st_real_hits3=("hits3", "mean"),
        st_real_mrr=("mrr", "mean"),
        st_real_hits1_sd=("hits1", "std"),
    )
)

shuffletrain_seed = st_matched.merge(
    st_real,
    on=["battery_id", "battery", "seed"],
)

paired_seed = realtrain_seed.merge(
    shuffletrain_seed,
    on=["battery_id", "battery", "seed"],
)

for metric in ["hits1", "hits3", "mrr"]:
    paired_seed[f"A_rt_real_minus_perm_{metric}"] = (
        paired_seed[f"rt_real_{metric}"]
        - paired_seed[f"rt_perm_{metric}"]
    )
    paired_seed[f"B_rt_real_minus_noedge_{metric}"] = (
        paired_seed[f"rt_real_{metric}"]
        - paired_seed[f"rt_noedge_{metric}"]
    )
    paired_seed[f"C_st_matched_minus_real_{metric}"] = (
        paired_seed[f"st_matched_{metric}"]
        - paired_seed[f"st_real_{metric}"]
    )

value_cols = [
    c for c in paired_seed.columns
    if c not in {"battery_id", "battery", "seed"}
]

fold_mean = (
    paired_seed
    .groupby(["battery_id", "battery"], as_index=False)[value_cols]
    .mean()
)

print("Seed-level paired rows:", len(paired_seed))
print("Held-out product rows:", len(fold_mean))
display(fold_mean)


## 17. E1 product-level inference

The exact sign-flip test is the inferential test. Bootstrap confidence intervals are descriptive. Holm correction is applied only to the three Hits@1 E1 contrasts A--C. Hits@3 and MRR are supporting endpoints and retain unadjusted exact p-values.


In [ ]:
# 17. Exact product-level E1 statistics

contrast_defs = [
    (
        "A",
        "Real-trained: real test - permutation-test mean",
        "A_rt_real_minus_perm",
    ),
    (
        "B",
        "Real-trained: real test - no-edge test",
        "B_rt_real_minus_noedge",
    ),
    (
        "C",
        "Permutation-trained: matched-permutation test - real test",
        "C_st_matched_minus_real",
    ),
]

stat_rows = []
bootstrap_seed_base = 20261220

for metric_idx, metric in enumerate(["hits1", "hits3", "mrr"]):
    for contrast_idx, (cid, label, prefix) in enumerate(contrast_defs):
        values = fold_mean[f"{prefix}_{metric}"].to_numpy(float)

        ci = bootstrap_mean_ci(
            values,
            n_boot=20000,
            seed=bootstrap_seed_base + 100 * metric_idx + contrast_idx,
        )
        p = exact_two_sided_sign_flip(values)

        stat_rows.append({
            "metric": metric,
            "contrast_id": cid,
            "contrast": label,
            "mean_difference": float(np.mean(values)),
            "ci_low": ci[0],
            "ci_high": ci[1],
            "positive_products": int(np.sum(values > 0)),
            "negative_products": int(np.sum(values < 0)),
            "zero_products": int(np.sum(values == 0)),
            "p_exact": float(p),
        })

contrast_stats = pd.DataFrame(stat_rows)
contrast_stats["p_holm_hits1"] = np.nan

h1_mask = contrast_stats["metric"] == "hits1"
contrast_stats.loc[h1_mask, "p_holm_hits1"] = holm_adjust(
    contrast_stats.loc[h1_mask, "p_exact"].to_numpy(float)
)

display(contrast_stats)

aggregate = pd.DataFrame([
    {
        "condition": "Real-trained / real test",
        "hits1": fold_mean["rt_real_hits1"].mean(),
        "hits3": fold_mean["rt_real_hits3"].mean(),
        "mrr": fold_mean["rt_real_mrr"].mean(),
    },
    {
        "condition": "Real-trained / permutation-test mean",
        "hits1": fold_mean["rt_perm_hits1"].mean(),
        "hits3": fold_mean["rt_perm_hits3"].mean(),
        "mrr": fold_mean["rt_perm_mrr"].mean(),
    },
    {
        "condition": "Real-trained / no-edge test",
        "hits1": fold_mean["rt_noedge_hits1"].mean(),
        "hits3": fold_mean["rt_noedge_hits3"].mean(),
        "mrr": fold_mean["rt_noedge_mrr"].mean(),
    },
    {
        "condition": "Permutation-trained / matched-permutation test mean",
        "hits1": fold_mean["st_matched_hits1"].mean(),
        "hits3": fold_mean["st_matched_hits3"].mean(),
        "mrr": fold_mean["st_matched_mrr"].mean(),
    },
    {
        "condition": "Permutation-trained / real test mean",
        "hits1": fold_mean["st_real_hits1"].mean(),
        "hits3": fold_mean["st_real_hits3"].mean(),
        "mrr": fold_mean["st_real_mrr"].mean(),
    },
])

display(aggregate)

print("\nInterpretation reminder:")
print(
    "A>0: real-trained models benefit from correct semantic-topological "
    "alignment on the held-out product."
)
print(
    "B>0: real-trained models rely on graph-branch cross-node information "
    "at inference relative to edge removal."
)
print(
    "C>0: permutation-trained models prefer their matched permutation "
    "over a test-time switch to the real graph; this diagnoses topology "
    "mismatch sensitivity, not physical correctness by itself."
)


## 18. Save E1 outputs

Send the generated ZIP back for interpretation. The most important files are:

- `test_time_topology_swap_eval_results.csv`
- `test_time_topology_swap_paired_seed.csv`
- `test_time_topology_swap_fold_mean.csv`
- `test_time_topology_swap_contrasts.csv`
- `test_time_topology_swap_aggregate.csv`
- `test_time_topology_swap_metadata.json`

The checkpoint folder is intentionally not included in the lightweight results ZIP.


In [ ]:
# 18. Save E1 outputs and create a lightweight results ZIP

swap_results.to_csv(
    OUT / "test_time_topology_swap_eval_results.csv",
    index=False,
)
paired_seed.to_csv(
    OUT / "test_time_topology_swap_paired_seed.csv",
    index=False,
)
fold_mean.to_csv(
    OUT / "test_time_topology_swap_fold_mean.csv",
    index=False,
)
contrast_stats.to_csv(
    OUT / "test_time_topology_swap_contrasts.csv",
    index=False,
)
aggregate.to_csv(
    OUT / "test_time_topology_swap_aggregate.csv",
    index=False,
)

metadata = {
    "experiment": "E1 test-time topology swap",
    "training_fits": int(10 * len(SEEDS) * (1 + len(SHUFFLE_IDS))),
    "neural_training_seeds": [int(x) for x in SEEDS],
    "permutation_ids": [int(x) for x in SHUFFLE_IDS],
    "epochs": int(EPOCHS),
    "negatives_per_state": int(NEGATIVES_PER_STATE),
    "learning_rate": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "dropout": float(DROPOUT),
    "bootstrap_replicates": 20000,
    "primary_e1_endpoint": "Hits@1",
    "independent_unit": "held-out battery product (n=10)",
    "notes": (
        "Canonical models are reproduced because the original experiment "
        "did not persist checkpoints. Graph swaps are performed only after "
        "training. Hits@1 contrasts A-C use exact paired sign-flip tests "
        "with Holm adjustment."
    ),
}

with open(
    OUT / "test_time_topology_swap_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

LIGHT = Path("/content/test_time_topology_swap_results_light")
if LIGHT.exists():
    shutil.rmtree(LIGHT)
LIGHT.mkdir(parents=True, exist_ok=True)

for filename in [
    "test_time_topology_swap_eval_results.csv",
    "test_time_topology_swap_paired_seed.csv",
    "test_time_topology_swap_fold_mean.csv",
    "test_time_topology_swap_contrasts.csv",
    "test_time_topology_swap_aggregate.csv",
    "test_time_topology_swap_metadata.json",
]:
    shutil.copy2(OUT / filename, LIGHT / filename)

zip_path = shutil.make_archive(
    "/content/test_time_topology_swap_results",
    "zip",
    root_dir=LIGHT,
)

print("Saved E1 outputs to:", OUT)
print("Lightweight results ZIP:", zip_path)
print("\nFiles to send back:")
for p in sorted(LIGHT.iterdir()):
    print(" -", p.name)
